# Stage 3 — Itinerary Builder (K-Means++ + TSP)
**[STUDENT VERSION — fill in the blanks]**

- Stage 3a: K-Means++ — gom điểm gần nhau vào cùng ngày
- Stage 3b: TSP Nearest Neighbor — tối ưu thứ tự đi trong ngày

> 💡 Cells marked `# TODO` require you to fill in the code.


In [ ]:
import csv, json, math, random
from pathlib import Path
import pandas as pd


In [ ]:
BASE_DIR       = Path('.')
STAGE2_CSV     = BASE_DIR.parent / 'stage2' / 'stage2_results.csv'
DATASET_CSV    = BASE_DIR.parent / 'stage1' / 'stage1_dataset.csv'
CONFIG_CSV     = BASE_DIR / 'stage3_config.csv'
ITINERARY_CSV  = BASE_DIR / 'stage3_itinerary.csv'
ITINERARY_JSON = BASE_DIR / 'stage3_itinerary.json'

def read_csv(path):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

stage2_rows = read_csv(STAGE2_CSV)
dataset     = read_csv(DATASET_CSV)
config_rows = read_csv(CONFIG_CSV)
print(f'Stage 2 rows: {len(stage2_rows)}')
print(f'Dataset: {len(dataset)}')


Stage 2 rows: 25
Dataset: 38


In [ ]:
# Distance helpers (đã có sẵn, không cần sửa)
def euclidean(a, b):
    """Euclidean distance on (lat,lng) — dùng cho K-Means và TSP so sánh."""
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def haversine_km(a, b):
    """Khoảng cách km thực tế (đường chim bay) — dùng để hiển thị output."""
    R=6371; dlat=math.radians(b[0]-a[0]); dlng=math.radians(b[1]-a[1])
    x=math.sin(dlat/2)**2+math.cos(math.radians(a[0]))*math.cos(math.radians(b[0]))*math.sin(dlng/2)**2
    return R*2*math.atan2(math.sqrt(x),math.sqrt(1-x))


## 🔧 TODO 1 — K-Means++ Initialization

K-Means++ khác standard K-Means ở bước chọn centroid ban đầu.

**Standard K-Means:** chọn K centroid ngẫu nhiên → dễ hội tụ xấu.

**K-Means++:** chọn centroid theo xác suất ∝ d² (điểm xa centroid hiện tại được chọn nhiều hơn):
```
Centroid 1: điểm đầu tiên
Centroid 2: sample với P ∝ d²(x, centroid_1)
Centroid 3: sample với P ∝ d²(x, nearest centroid)
...
```

Điền phần init vào hàm `kmeans_plus_plus()` bên dưới.


In [ ]:
def kmeans_plus_plus(points, k, n_iter=60):
    """
    K-Means++ clustering trên (lat, lng).
    points: list of (lat, lng)
    k: số ngày = số cụm
    Returns: list of k lists of indices
    """
    if len(points) <= k:
        return [[i] for i in range(len(points))]

    # TODO 1a: K-Means++ init
    centroids = [points[0]]  # centroid đầu tiên = điểm đầu tiên

    while len(centroids) < k:
        # Tính d² từ mỗi điểm đến centroid gần nhất
        d2    = None  # ← [min(euclidean(p,c)**2 for c in centroids) for p in points]
        total = None  # ← sum(d2)

        # Roulette wheel selection theo xác suất tỉ lệ với d²
        r      = random.uniform(0, total)
        cumsum = 0.0
        for i, d in enumerate(d2):
            cumsum += d
            if cumsum >= r:
                centroids.append(points[i])
                break
        else:
            centroids.append(points[-1])

    # Assign + Recompute (đã có sẵn)
    clusters = None
    for _ in range(n_iter):
        clusters = [[] for _ in range(k)]
        for i, p in enumerate(points):
            nearest = min(range(k), key=lambda ci: euclidean(p, centroids[ci]))
            clusters[nearest].append(i)
        new_c = []
        for ci in range(k):
            if clusters[ci]:
                new_c.append((sum(points[i][0] for i in clusters[ci])/len(clusters[ci]),
                              sum(points[i][1] for i in clusters[ci])/len(clusters[ci])))
            else:
                new_c.append(centroids[ci])
        if new_c == centroids: break
        centroids = new_c

    return clusters

print('kmeans_plus_plus() defined.')


## 🔧 TODO 2 — TSP Nearest Neighbor (multi-start)

Nearest Neighbor heuristic:
1. Xuất phát từ 1 điểm
2. Luôn đi đến điểm **chưa thăm gần nhất**
3. Lặp đến hết

**Multi-start:** Thử tất cả N điểm xuất phát, lấy route ngắn nhất.

Điền phần tìm điểm gần nhất và tính tổng khoảng cách.


In [ ]:
def tsp_nearest_neighbor(points):
    """Multi-start Nearest Neighbor TSP. Returns list of indices."""
    n = len(points)
    if n <= 1: return list(range(n))

    best_order, best_dist = None, float('inf')

    for start in range(n):
        visited, order, cur = {start}, [start], start
        while len(order) < n:
            # TODO 2a: Tìm điểm chưa thăm gần nhất
            nxt = None  # ← min((i for i in range(n) if i not in visited),
                        #        key=lambda i: euclidean(points[cur], points[i]))
            visited.add(nxt); order.append(nxt); cur = nxt

        # TODO 2b: Tính tổng khoảng cách Haversine của route này
        total = None  # ← sum(haversine_km(points[order[i]], points[order[i+1]])
                      #        for i in range(len(order)-1))

        if total < best_dist:
            best_dist = total; best_order = order

    return best_order

print('tsp_nearest_neighbor() defined.')


## 🔧 TODO 3 — `build_itinerary()` — Kết hợp Stage 3A + 3B

In [ ]:
def build_itinerary(selected_places, n_days):
    """
    K-Means++ phân ngày → TSP tối ưu thứ tự trong ngày.
    Returns: list of day dicts
    """
    k      = min(n_days, len(selected_places))
    coords = [(p['lat'], p['lng']) for p in selected_places]

    # TODO 3a: Gọi K-Means++ để phân cụm
    clusters = None  # ← kmeans_plus_plus(coords, k)

    itinerary = []
    for day_idx, ci in enumerate(clusters):
        if not ci: continue
        day_places = [selected_places[i] for i in ci]
        day_coords = [(p['lat'], p['lng']) for p in day_places]

        # TODO 3b: Gọi TSP để sắp xếp thứ tự
        order  = None  # ← tsp_nearest_neighbor(day_coords)
        sorted_day = [day_places[i] for i in order]

        # Tính khoảng cách giữa các stop
        legs, total = [], 0.0
        for i in range(len(sorted_day)-1):
            km = haversine_km((sorted_day[i]['lat'],   sorted_day[i]['lng']),
                              (sorted_day[i+1]['lat'], sorted_day[i+1]['lng']))
            legs.append(round(km,1)); total += km

        itinerary.append({'day':day_idx+1,'stops':sorted_day,
                          'legs_km':legs,'total_km':round(total,1)})
    return itinerary

print('build_itinerary() defined.')


In [ ]:
# Run + Save (đã có sẵn, không cần sửa)
coord_map = {row['place']:{'lat':float(row['lat']),'lng':float(row['lng']),
             'province':row['province']} for row in dataset}
stage2_by_profile = {}
for row in stage2_rows:
    stage2_by_profile.setdefault(row['profile'],[]).append(row)
config = {r['profile_name']:{'n_days':int(r['n_days']),'top_k':int(r['top_k'])} for r in config_rows}

all_results, itinerary_rows = [], []
for profile_name, cfg in config.items():
    n_days, top_k = cfg['n_days'], cfg['top_k']
    s2 = sorted(stage2_by_profile.get(profile_name,[]),key=lambda x:int(x['rank']))[:top_k]
    if not s2: continue
    selected = [{'name':r['place'],'province':coord_map[r['place']]['province'],
                 'lat':coord_map[r['place']]['lat'],'lng':coord_map[r['place']]['lng'],
                 'cosine_score':float(r['cosine_score']),'month_match':r['month_match']}
                for r in s2 if r['place'] in coord_map]
    print(f'\nProfile: {profile_name} | {len(selected)} places | {n_days} days')
    itin = build_itinerary(selected, n_days)
    all_results.append({'profile':profile_name,'n_days':n_days,'itinerary':itin})
    for day in itin:
        for si,stop in enumerate(day['stops']):
            leg = day['legs_km'][si] if si<len(day['legs_km']) else 0.0
            itinerary_rows.append({'profile':profile_name,'day':day['day'],'stop_order':si+1,
                'place':stop['name'],'province':stop['province'],'leg_km':leg,'day_total_km':day['total_km']})
        print(f"  Day {day['day']}: {' → '.join(s['name'] for s in day['stops'])} ({day['total_km']} km)")

headers=['profile','day','stop_order','place','province','leg_km','day_total_km']
with ITINERARY_CSV.open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=headers); w.writeheader(); w.writerows(itinerary_rows)
with ITINERARY_JSON.open('w',encoding='utf-8') as f:
    json.dump(all_results,f,ensure_ascii=False,indent=2)
print(f'\nSaved: {ITINERARY_CSV.name}')
print(f'Saved: {ITINERARY_JSON.name}')
